# Team stats cleaning

In [1]:
import pandas as pd
import os

In [2]:
RAW_PATH = "../../data/teams"
CLEANED_PATH = "../../data/teams_cleaned"
os.makedirs(CLEANED_PATH, exist_ok=True)

### Step 1: Load and preview team data

In [3]:
def load_and_merge_team_data(start_year = 2010, end_year = 2025 ):
    team_df = []

    for year in range(start_year, end_year +1):
        file_path = f"{RAW_PATH}/team_combined_stats_{year}.csv"
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            df["Season"] = year
            team_df.append(df)
        else:
            print(f"Missing: {file_path}")
    
    team_merged = pd.concat(team_df, ignore_index=True)
    return team_merged

In [4]:
team_merged = load_and_merge_team_data()

Missing: ../../data/teams/team_combined_stats_2025.csv


Previewing the dataset after merging. Here we'll be checking for missing values, columns that may not be needed and anything that stands out.

In [5]:
def describe_dataframe(df):
    print("  TYPES   ".center(82, '-'))
    print(df.dtypes)

    print("  SHAPE   ".center(82,'-'))
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    print("  DUPLICATES  ".center(82,'-'))
    print(df.duplicated().sum())

    print("  NaN VALUES".center(82,'-'))
    print(df.isna().sum())

    #print("  QUANTILES   ".center(82,'-'))
    #print(df.quantile([0,0.05,0.50,0.95,0.99,1]).T)

    print("  DATAFRAME HEAD      ".center(82, '-'))
    print(df.head(n=5))
    print("  DATAFRAME TAIL      ".center(82, '-'))
    print(df.tail(n=5))

In [6]:
describe_dataframe(team_merged)

------------------------------------  TYPES   ------------------------------------
Team                    object
G                        int64
MP                     float64
FG                     float64
FGA                    float64
FG%                    float64
3P                     float64
3PA                    float64
3P%                    float64
2P                     float64
2PA                    float64
2P%                    float64
FT                     float64
FTA                    float64
FT%                    float64
ORB                    float64
DRB                    float64
TRB                    float64
AST                    float64
STL                    float64
BLK                    float64
TOV                    float64
PF                     float64
PTS                    float64
Age                    float64
W                      float64
L                      float64
PW                       int64
PL                       int64
MOV               

Dropping columns that are not relevant for making inferences about team positioning in the standings. Columns "G" and "MP" are only relevant to determing individual player awards since all teams play the same number of games in a regular season. "Attend." and "Attend./G" are also not needed for making assumptions. "Age" of a team may be a determining factor in team performance, but this is harder to measure. Typically in the NBA younger teams tend to have worse seasons than older teams, but the correlation isn't high enough to be worth measuring or exploring. 

In [7]:
columns_to_drop = ["Arena", "Attend.", "Attend./G", "Age", "G", 
                   "MP", "Unnamed: 27_level_1", "Unnamed: 22_level_1", 
                   "Unnamed: 17_level_1", "PW", "PL", "eFG%.1", "TOV%.1", "FT/FGA.1"]

team_merged.drop(columns=columns_to_drop, inplace=True)

In [8]:
# Reviewing the dataframe after dropping columns
team_merged.head(5)

,Team,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,...,Pace,FTr,3PAr,TS%,eFG%,TOV%,ORB%,FT/FGA,DRB%,Season
0,Phoenix Suns*,40.7,82.8,0.492,8.9,21.6,0.412,31.8,61.2,0.520,...,95.3,0.312,0.261,0.585,0.546,13.6,27.6,0.240,70.8,2010
1,Golden State Warriors,40.6,86.5,0.469,7.7,20.6,0.375,32.9,65.9,0.499,...,100.4,0.294,0.238,0.557,0.514,13.1,20.9,0.230,68.5,2010
2,Denver Nuggets*,38.1,81.4,0.468,6.6,18.5,0.359,31.5,62.9,0.500,...,94.8,0.376,0.227,0.561,0.509,12.7,26.1,0.290,72.4,2010
3,Utah Jazz*,39.4,80.2,0.491,5.4,14.7,0.364,34.0,65.5,0.519,...,93.8,0.340,0.184,0.565,0.524,14.2,26.8,0.252,75.6,2010
4,Toronto Raptors,39.0,80.9,0.482,6.3,17.0,0.371,32.7,63.8,0.512,...,93.1,0.319,0.211,0.564,0.521,12.7,24.7,0.244,72.9,2010


Going through the DataFrame showed there are some variations to the team names throughout. So we're going to dive deeper into the "Team" column of the team_merged DataFrame to understand what's going on.

In [9]:
print(f"Number of unique teams: {team_merged["Team"].nunique()}")

Number of unique teams: 65


The number of unique teams count is off due to a trailing "*", this will have to be removed to avoid any issues further down the line. The count of unique teams should be exactly 30 as there are 30 teams in the league.

In [10]:
team_merged["Team"] = team_merged["Team"].str.strip("*")

In [11]:
# Checking the number of unique teams
print(f"Number of unique teams: {team_merged["Team"].nunique()}")

Number of unique teams: 33


There are still 3 more teams being found in the dataset than there are teams in the league. Now we'll investigate by printing all unique names.

In [12]:
print(team_merged["Team"].unique())

['Phoenix Suns' 'Golden State Warriors' 'Denver Nuggets' 'Utah Jazz'
 'Toronto Raptors' 'Orlando Magic' 'Memphis Grizzlies' 'Houston Rockets'
 'Cleveland Cavaliers' 'New York Knicks' 'Dallas Mavericks'
 'Los Angeles Lakers' 'Atlanta Hawks' 'Oklahoma City Thunder'
 'San Antonio Spurs' 'Indiana Pacers' 'New Orleans Hornets'
 'Sacramento Kings' 'Boston Celtics' 'Minnesota Timberwolves'
 'Portland Trail Blazers' 'Milwaukee Bucks' 'Philadelphia 76ers'
 'Chicago Bulls' 'Miami Heat' 'Washington Wizards' 'Los Angeles Clippers'
 'Charlotte Bobcats' 'Detroit Pistons' 'New Jersey Nets' 'Brooklyn Nets'
 'New Orleans Pelicans' 'Charlotte Hornets']


After printing the names we can see the issue. Three teams changed their names over the last 15 years, so to resolve this issue we need to rename the rows as follows:
"New Orleans Hornets" = "New Orleans Pelicans"
"Charlotte Bobcats" = "Charlotte Hornets"
"New Jersey Nets" = "Brooklyn Nets"

In [13]:
team_merged["Team"] = team_merged["Team"].replace("Charlotte Bobcats", "Charlotte Hornets")
team_merged["Team"] = team_merged["Team"].replace("New Orleans Hornets", "New Orleans Pelicans")
team_merged["Team"] = team_merged["Team"].replace("New Jersey Nets", "Brooklyn Nets")

In [14]:
# Checking the count of team names
print(f"Number of unique teams: {team_merged["Team"].nunique()}")

Number of unique teams: 30


In [15]:
# Checking team names
print(team_merged["Team"].unique())

['Phoenix Suns' 'Golden State Warriors' 'Denver Nuggets' 'Utah Jazz'
 'Toronto Raptors' 'Orlando Magic' 'Memphis Grizzlies' 'Houston Rockets'
 'Cleveland Cavaliers' 'New York Knicks' 'Dallas Mavericks'
 'Los Angeles Lakers' 'Atlanta Hawks' 'Oklahoma City Thunder'
 'San Antonio Spurs' 'Indiana Pacers' 'New Orleans Pelicans'
 'Sacramento Kings' 'Boston Celtics' 'Minnesota Timberwolves'
 'Portland Trail Blazers' 'Milwaukee Bucks' 'Philadelphia 76ers'
 'Chicago Bulls' 'Miami Heat' 'Washington Wizards' 'Los Angeles Clippers'
 'Charlotte Hornets' 'Detroit Pistons' 'Brooklyn Nets']


Now that the data has been cleaned, it is ready for the EDA phase.

In [16]:
team_merged.to_csv(f"{CLEANED_PATH}/team_all_years.csv", index=False)
print(f"Saved cleaned team dataset to: {CLEANED_PATH}/team_all_years.csv")

Saved cleaned team dataset to: ../../data/teams_cleaned/team_all_years.csv
